# MODFLOW 6 API C: changing recharge with a callback

This is notebook C of the API series. Notebook A ([`mf6-api-a-basic`](mf6-api-a-basic.ipynb)) covers the steps of an API run.

Change an input while the model runs. A **callback** is a function that `modflowapi.run_simulation()` calls at set points of the run. Here it raises the recharge of the synthetic-valley base model by 10 percent at the start of every stress period, as a wetter climate would, without changing the input files. The water budget of the run then shows whether the change took.

## What this notebook covers

By the end of this notebook you will be able to:

- write a callback and pass it to `modflowapi.run_simulation()`,
- change the recharge at the start of every stress period without changing the input files, and
- check the change against the water budget of the run.

## Imports

Import FloPy, NumPy, pandas, and Matplotlib, and `Callbacks` and `run_simulation` from `modflowapi`. `mf6_api_helpers` finds the MODFLOW 6 library, and `BASE_MODEL` from `sv_notebook_helpers` is the folder of the synthetic-valley base model.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mf6_api_helpers import find_mf6_libraries
from modflowapi import Callbacks, run_simulation
from sv_notebook_helpers import BASE_MODEL

## Find the MODFLOW 6 library

The API runs MODFLOW 6 through its shared library, `libmf6` (`libmf6.so` on Linux, `libmf6.dylib` on macOS, and `libmf6.dll` on Windows), rather than through the `mf6` program. pixi installs both with the class MODFLOW 6. `find_mf6_libraries()` returns the path of each.

In [ ]:
lib_path, mf6_exe = find_mf6_libraries()
print(f"library: {lib_path.name}")
print(f"program: {mf6_exe.name}")

## Write the model

Load the base model and write it to a new workspace. Its Recharge package, named `rch_0`, lists a recharge rate in meters per day (m/d) for every land and lake cell of layer 1, in each of the 21 stress periods.

In [ ]:
name = "sv"
ws = pl.Path("models/mf6-api-c-recharge")

sim = flopy.mf6.MFSimulation.load(
    sim_ws=BASE_MODEL, exe_name=mf6_exe, write_headers=False, verbosity_level=0
)
sim.set_sim_path(ws)
sim.write_simulation(silent=True)
gwf = sim.get_model(name)

## The callback

`run_simulation()` calls the callback with two arguments: the running simulation, and the point of the run it has reached, one of the `Callbacks` values such as `Callbacks.stress_period_start`, `Callbacks.timestep_start`, or `Callbacks.iteration_start`. The running simulation holds each model by name, `sim.sv` here, and each model holds its packages by name, so `sim.sv.rch_0` is the Recharge package. Its `stress_period_data` is the data of the current stress period, a NumPy record array whose `recharge` column can be changed in place. The change goes straight into the running model.

At the start of each stress period, MODFLOW reads that period's recharge from the input, and the callback then multiplies it by 1.1. A callback like this needs no memory addresses, because `modflowapi` gives the standard stress packages by name. Notebooks D and E read and write variables the packages do not give, through `get_value()` and `get_value_ptr()`.

In [ ]:
RECHARGE_FACTOR = 1.1


def callback(sim, step):
    # raise the recharge of every stress period as it starts
    if step == Callbacks.stress_period_start:
        sim.sv.rch_0.stress_period_data["recharge"] *= RECHARGE_FACTOR

In [ ]:
run_simulation(lib_path, ws, callback, verbose=False)

## Check the change

Compare the recharge the input files give with the recharge the run reports. The input rate of a cell is its recharge in m/d times its area, from `gwf.modelgrid.area`, and summing the cells of each stress period gives the input recharge in cubic meters per day (m³/d). The run wrote its water budget to `sv-budget.csv`, which `gwf.output.budgetcsv().data` reads; the column `RCH(RCH_0)_IN` is the recharge the model applied. Convert both to million cubic meters per year (million m³/yr), as in the synthetic-valley notebooks.

In [ ]:
M3D_TO_MCM_PER_YEAR = 365.25 / 1.0e6
# the area of each cell, by row and column
area = gwf.modelgrid.area.reshape(gwf.modelgrid.nrow, gwf.modelgrid.ncol)
nper = sim.tdis.nper.data

recharge_input = []
for kper in range(nper):
    data = gwf.rch.stress_period_data.get_data(kper)
    rows = [cellid[1] for cellid in data["cellid"]]
    cols = [cellid[2] for cellid in data["cellid"]]
    recharge_input.append((data["recharge"] * area[rows, cols]).sum())

budget = gwf.output.budgetcsv().data
recharge = pd.DataFrame(
    {
        "input": recharge_input,
        "simulated": budget["RCH(RCH_0)_IN"],
    },
    index=pd.Index(["Steady state"] + list(range(2006, 2026)), name="period"),
)
recharge = recharge * M3D_TO_MCM_PER_YEAR
recharge["ratio"] = recharge["simulated"] / recharge["input"]
recharge.round(3)

Plot the input and simulated recharge of each year of the transient periods as bars side by side.

In [ ]:
width = 17.15 / 2.54  # figure width, inches
years = np.array(recharge.index[1:], dtype=int)

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.45 * width), layout="constrained")
    ax.bar(
        years - 0.2,
        recharge["input"].iloc[1:],
        width=0.4,
        color="0.6",
        label="Input files",
    )
    ax.bar(
        years + 0.2,
        recharge["simulated"].iloc[1:],
        width=0.4,
        color="#2a78d6",
        label="Simulated, with the callback",
    )
    flopy.plot.styles.heading(ax=ax, heading="Recharge by year")
    flopy.plot.styles.xlabel(ax=ax, label="Year")
    flopy.plot.styles.ylabel(ax=ax, label="Recharge (million m³/yr)")
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    fig.legend(loc="outside lower center", ncols=2, frameon=False)

**What to look for.** The simulated recharge is 1.1 times the recharge in the input files in every stress period, the steady-state period included, so the callback acted at the start of each one. In 2018, the year with the least recharge, it raises 15.44 million m³/yr to 16.98 million m³/yr; in the other years it raises 20.71 to 23.70 million m³/yr to 22.78 to 26.07 million m³/yr. The input files are unchanged, so running the model with the `mf6` program gives the original recharge.

## Recap

You wrote a callback that multiplies the recharge of the synthetic-valley base model by 1.1 at the start of every stress period, ran the model with it through `run_simulation()`, and checked the water budget of the run against the input files. A callback reaches the standard stress packages by name, so changing them needs no knowledge of how MODFLOW stores them.